# Miniproyecto 3: Conversión de Escenas Diurnas a Nocturnas con CycleGAN
**Curso:** Redes Neuronales y Deep Learning — Universidad Autónoma de Occidente  
**Objetivo:** Implementar y entrenar una red generativa adversarial cíclica (**CycleGAN**) para transferir el estilo de escenas urbanas entre el **Dominio A (Día)** y el **Dominio B (Noche)** sin necesidad de imágenes emparejadas (*unpaired image-to-image translation*), utilizando un conjunto curado de 200 imágenes por dominio.

### Técnicas Avanzadas Incorporadas para Evitar Sobreentrenamiento y Mejorar la Calidad:
1. **Regulación y Estabilización Adversarial (de *Gan_RED.ipynb*):**
   - **Label Smoothing unilateral (0.9 para etiquetas reales):** Suaviza las probabilidades del discriminador evitando gradientes extremos y colapso de modo.
   - **Dropout espacial (0.2 - 0.25) en el Discriminador PatchGAN:** Previene la memorización de texturas específicas y asegura un aprendizaje generalizado.
   - **Buffer de Replay de Imágenes Falsas (*History Image Buffer*, tamaño 50):** Rompe la correlación temporal y estabiliza el entrenamiento del discriminador evitando oscilaciones caóticas.
   - **LeakyReLU ($\alpha = 0.2$) y Optimizadores Adam ($\beta_1 = 0.5$):** Configuración estándar de GANs estables.
2. **Alineación Estructural y Regularización (de *Autoendores_function.ipynb*):**
   - **Pérdida de Consistencia Cíclica $\mathcal{L}_{cycle}$ (tipo Autoencoder bidireccional):** $F(G(x)) \approx x$ y $G(F(y)) \approx y$ actúa como un autocodificador cruzado profundo que preserva la geometría y la estructura espacial de edificios, calles y cielo.
   - **Pérdida de Identidad $\mathcal{L}_{identity}$:** $G(y) \approx y$ y $F(x) \approx x$ preserva el espacio de color intrínseco.
   - **Decay Progresivo del Learning Rate (*Schedulers*):** Reduce linealmente la tasa de aprendizaje desde la época 20 hasta la época 45 para converger suavemente hacia un óptimo sin sobreajuste.
3. **Data Augmentation Dinámico en TensorFlow:**
   - *Random Jittering* (escalado a 286x286 + recorte aleatorio a 256x256 por cada época sin caché prematuro).
   - *Random Mirroring* horizontal y normalización en $[-1, 1]$ con activación `tanh` en el generador.

In [ ]:
import os
import shutil
import subprocess

if os.path.basename(os.getcwd()) != "RepositorioMiniproyecto3":
    if os.path.exists("RepositorioMiniproyecto3"):
        shutil.rmtree("RepositorioMiniproyecto3")
    subprocess.run(["git", "clone", "https://github.com/AlanBas18/RepositorioMiniproyecto3.git"], check=True)

import time
import glob
import random
import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageOps
from tqdm.auto import tqdm
from IPython.display import clear_output, display

import tensorflow as tf
from tensorflow.keras import layers, Model

print("TensorFlow Version:", tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
print("GPU(s) disponible(s):", gpus if gpus else "Ninguna detectada (Revisa Entorno de ejecución > Cambiar tipo de entorno > T4 GPU)")

## 1. Carga, Filtrado a 200 Imágenes y Data Augmentation Dinámico
Para entrenar eficazmente el modelo con el subconjunto representativo de **200 imágenes para `trainA`** y **200 imágenes para `trainB`** (junto con el conjunto de evaluación `testA` y `testB`):
* Se detecta automáticamente la raíz del repositorio (en Google Colab o en entorno local).
* Se toman exactamente **200 imágenes de entrenamiento** por dominio para asegurar balance estricto.
* Se aplica **aumento dinámico de datos** (*Data Augmentation*) en cada época: escalado a `286x286`, recorte aleatorio a `256x256` y volteo horizontal aleatorio.
* **Sin `.cache()` antes del aumento** para que cada época presente variantes geométricas nuevas a la red, evitando que los modelos memoricen las 200 imágenes.

In [ ]:
if os.path.exists('/content/RepositorioMiniproyecto3/dataset'):
    REPO_DIR = '/content/RepositorioMiniproyecto3'
elif os.path.exists('./RepositorioMiniproyecto3/dataset'):
    REPO_DIR = './RepositorioMiniproyecto3'
else:
    REPO_DIR = '.'

BASE_DIR = os.path.join(REPO_DIR, 'dataset')

PATH_TRAIN_A = os.path.join(BASE_DIR, 'trainA')
PATH_TRAIN_B = os.path.join(BASE_DIR, 'trainB')
PATH_TEST_A  = os.path.join(BASE_DIR, 'testA')
PATH_TEST_B  = os.path.join(BASE_DIR, 'testB')

BUFFER_SIZE = 1000
BATCH_SIZE = 1
IMG_WIDTH = 256
IMG_HEIGHT = 256
MAX_TRAIN_SAMPLES = 200

def load_image(image_file):
    image = tf.io.read_file(image_file)
    image = tf.io.decode_image(image, channels=3, expand_animations=False)
    image = tf.cast(image, tf.float32)
    return image

def normalize(image):
    return (image / 127.5) - 1.0

def random_crop(image):
    return tf.image.random_crop(image, size=[IMG_HEIGHT, IMG_WIDTH, 3])

def random_jitter(image):
    image = tf.image.resize(image, [286, 286], method=tf.image.ResizeMethod.NEAREST_NEIGHBOR)
    image = random_crop(image)
    image = tf.image.random_flip_left_right(image)
    return image

def preprocess_image_train(image_file):
    image = load_image(image_file)
    image = random_jitter(image)
    image = normalize(image)
    return image

def preprocess_image_test(image_file):
    image = load_image(image_file)
    image = tf.image.resize(image, [IMG_HEIGHT, IMG_WIDTH])
    image = normalize(image)
    return image

def crear_dataset(folder_path, is_train=True, max_samples=None):
    valid_exts = ('.jpg', '.jpeg', '.png')
    files = sorted([os.path.join(folder_path, f) for f in os.listdir(folder_path) if f.lower().endswith(valid_exts)])

    if max_samples is not None and len(files) > max_samples:
        files = files[:max_samples]

    ds = tf.data.Dataset.from_tensor_slices(files)
    if is_train:
        ds = ds.map(preprocess_image_train, num_parallel_calls=tf.data.AUTOTUNE)
        ds = ds.shuffle(BUFFER_SIZE).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    else:
        ds = ds.map(preprocess_image_test, num_parallel_calls=tf.data.AUTOTUNE)
        ds = ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    return ds, len(files)

train_dia, n_train_a = crear_dataset(PATH_TRAIN_A, is_train=True, max_samples=MAX_TRAIN_SAMPLES)
train_noche, n_train_b = crear_dataset(PATH_TRAIN_B, is_train=True, max_samples=MAX_TRAIN_SAMPLES)
test_dia, n_test_a = crear_dataset(PATH_TEST_A, is_train=False)
test_noche, n_test_b = crear_dataset(PATH_TEST_B, is_train=False)

print("=== Conjunto de Datos Configurado ===")
print(f"Entrenamiento -> Día   (trainA): {n_train_a} imágenes")
print(f"Entrenamiento -> Noche (trainB): {n_train_b} imágenes")
print(f"Evaluación    -> Día   (testA):  {n_test_a} imágenes")
print(f"Evaluación    -> Noche (testB):  {n_test_b} imágenes")

sample_dia = next(iter(train_dia))
sample_noche = next(iter(train_noche))

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.title('Dominio A: Escena Diurna Real (Input)')
plt.imshow(np.clip(sample_dia[0].numpy() * 0.5 + 0.5, 0.0, 1.0))
plt.axis('off')

plt.subplot(1, 2, 2)
plt.title('Dominio B: Escena Nocturna Real (Target Style)')
plt.imshow(np.clip(sample_noche[0].numpy() * 0.5 + 0.5, 0.0, 1.0))
plt.axis('off')
plt.tight_layout()
plt.show()

## 2. Arquitectura de Redes y Técnicas Anti-Sobreajuste

### Componentes:
1. **Instance Normalization (`InstanceNormalization`):** Normaliza las activaciones a nivel de cada imagen individual, evitando artefactos entre lotes y permitiendo una transferencia de estilo precisa. Tanto esta capa como `ReflectionPadding2D` incluyen `@tf.keras.utils.register_keras_serializable` y `get_config()` para permitir la exportación y recarga completa de los modelos en formato `.keras` sin errores de deserialización.
2. **Generador ResNet (6 Bloques Residuales):** Usa *Reflection Padding* (para evitar bordes oscuros o distorsionados) y *Bilinear Upsampling + Conv2D* en lugar de `Conv2DTranspose` para eliminar completamente los patrones de cuadrícula (*checkerboard artifacts*).
3. **Discriminador PatchGAN con Dropout Regulador (Técnica de *Gan_RED.ipynb*):**
   - Evalúa el realismo en parches locales de $70 \times 70$ píxeles usando `LeakyReLU(negative_slope=0.2)` (actualizado a la API vigente de Keras 3).
   - Se incorpora **Dropout (0.2 - 0.25)** antes de las capas de decisión para impedir que el discriminador sobreaprenda características espaciales rígidas de las 200 imágenes y supere injustamente al generador.

In [ ]:
@tf.keras.utils.register_keras_serializable(package="CycleGAN")
class InstanceNormalization(layers.Layer):
    def __init__(self, epsilon=1e-5, **kwargs):
        super().__init__(**kwargs)
        self.epsilon = epsilon

    def build(self, input_shape):
        self.scale = self.add_weight(
            name='scale',
            shape=(input_shape[-1],),
            initializer='ones',
            trainable=True
        )
        self.offset = self.add_weight(
            name='offset',
            shape=(input_shape[-1],),
            initializer='zeros',
            trainable=True
        )

    def call(self, x):
        mean, variance = tf.nn.moments(x, axes=[1, 2], keepdims=True)
        inv = tf.math.rsqrt(variance + self.epsilon)
        normalized = (x - mean) * inv
        return self.scale * normalized + self.offset

    def get_config(self):
        config = super().get_config()
        config.update({"epsilon": self.epsilon})
        return config

@tf.keras.utils.register_keras_serializable(package="CycleGAN")
class ReflectionPadding2D(layers.Layer):
    def __init__(self, padding=(1, 1), **kwargs):
        super().__init__(**kwargs)
        self.padding = tuple(padding)

    def call(self, input_tensor):
        padding_width, padding_height = self.padding
        return tf.pad(
            input_tensor,
            [[0, 0], [padding_height, padding_height], [padding_width, padding_width], [0, 0]],
            mode='REFLECT'
        )

    def get_config(self):
        config = super().get_config()
        config.update({"padding": self.padding})
        return config

def residual_block(x, filters, kernel_size=3):
    initializer = tf.random_normal_initializer(0., 0.02)
    input_tensor = x

    x = ReflectionPadding2D((1, 1))(x)
    x = layers.Conv2D(filters, kernel_size, padding='valid', kernel_initializer=initializer, use_bias=False)(x)
    x = InstanceNormalization()(x)
    x = layers.ReLU()(x)

    x = ReflectionPadding2D((1, 1))(x)
    x = layers.Conv2D(filters, kernel_size, padding='valid', kernel_initializer=initializer, use_bias=False)(x)
    x = InstanceNormalization()(x)

    return layers.Add()([input_tensor, x])

def build_resnet_generator(num_res_blocks=6, name="ResNet_Generator"):
    initializer = tf.random_normal_initializer(0., 0.02)
    inputs = layers.Input(shape=[256, 256, 3], name="input_image")

    x = ReflectionPadding2D((3, 3))(inputs)
    x = layers.Conv2D(64, 7, padding='valid', kernel_initializer=initializer, use_bias=False)(x)
    x = InstanceNormalization()(x)
    x = layers.ReLU()(x)

    for filters in [128, 256]:
        x = layers.Conv2D(filters, 3, strides=2, padding='same', kernel_initializer=initializer, use_bias=False)(x)
        x = InstanceNormalization()(x)
        x = layers.ReLU()(x)

    for _ in range(num_res_blocks):
        x = residual_block(x, 256)

    for filters in [128, 64]:
        x = layers.UpSampling2D(size=(2, 2), interpolation='bilinear')(x)
        x = ReflectionPadding2D((1, 1))(x)
        x = layers.Conv2D(filters, 3, padding='valid', kernel_initializer=initializer, use_bias=False)(x)
        x = InstanceNormalization()(x)
        x = layers.ReLU()(x)

    x = ReflectionPadding2D((3, 3))(x)
    outputs = layers.Conv2D(3, 7, padding='valid', kernel_initializer=initializer, activation='tanh')(x)

    return Model(inputs=inputs, outputs=outputs, name=name)

def build_patchgan_discriminator(name="PatchGAN_Discriminator"):
    initializer = tf.random_normal_initializer(0., 0.02)
    inputs = layers.Input(shape=[256, 256, 3], name="input_image")

    x = layers.Conv2D(64, 4, strides=2, padding='same', kernel_initializer=initializer)(inputs)
    x = layers.LeakyReLU(negative_slope=0.2)(x)

    x = layers.Conv2D(128, 4, strides=2, padding='same', kernel_initializer=initializer, use_bias=False)(x)
    x = InstanceNormalization()(x)
    x = layers.LeakyReLU(negative_slope=0.2)(x)
    x = layers.Dropout(0.2)(x)

    x = layers.Conv2D(256, 4, strides=2, padding='same', kernel_initializer=initializer, use_bias=False)(x)
    x = InstanceNormalization()(x)
    x = layers.LeakyReLU(negative_slope=0.2)(x)
    x = layers.Dropout(0.25)(x)

    x = layers.Conv2D(512, 4, strides=1, padding='same', kernel_initializer=initializer, use_bias=False)(x)
    x = InstanceNormalization()(x)
    x = layers.LeakyReLU(negative_slope=0.2)(x)
    x = layers.Dropout(0.25)(x)

    outputs = layers.Conv2D(1, 4, strides=1, padding='same', kernel_initializer=initializer)(x)

    return Model(inputs=inputs, outputs=outputs, name=name)

generator_g = build_resnet_generator(num_res_blocks=6, name="Generator_Day2Night")
generator_f = build_resnet_generator(num_res_blocks=6, name="Generator_Night2Day")
discriminator_x = build_patchgan_discriminator(name="Discriminator_Day")
discriminator_y = build_patchgan_discriminator(name="Discriminator_Night")

print("Arquitecturas creadas exitosamente:")
print(" - Generador G (Día -> Noche) parámetros:", f"{generator_g.count_params():,}")
print(" - Generador F (Noche -> Día) parámetros:", f"{generator_f.count_params():,}")
print(" - Discriminador X/Y parámetros (c/u):  ", f"{discriminator_y.count_params():,}")

## 3. Buffer de Historial de Imágenes, Funciones de Pérdida y Optimizadores

### Técnicas aplicadas:
1. **Replay Image Buffer (Capacidad 50):** El discriminador se entrena en un 50% con imágenes generadas recientes y en un 50% con imágenes generadas en iteraciones pasadas, evitando oscilaciones destructivas.
2. **Label Smoothing Unilateral:** Para las imágenes reales usamos target `0.9` en lugar de `1.0`. Esto previene que el discriminador sea excesivamente confiado y desestabilice los generadores.
3. **Ponderaciones $\lambda_{cycle} = 10.0$ y $\lambda_{identity} = 5.0$:** Fuerza a la red a mantener la fidelidad estructural y cromática.
4. **Decaimiento lineal del Learning Rate:** Mantiene `2e-4` hasta la época 20 y luego decrece linealmente hasta la época 45 para un ajuste fino estable.

In [ ]:
class ImagePool:
    def __init__(self, pool_size=50):
        self.pool_size = pool_size
        self.images = []

    def query(self, image):
        if self.pool_size == 0:
            return image
        if len(self.images) < self.pool_size:
            self.images.append(image)
            return image
        if random.random() > 0.5:
            idx = random.randint(0, self.pool_size - 1)
            tmp = self.images[idx]
            self.images[idx] = image
            return tmp
        return image

pool_fake_x = ImagePool(pool_size=50)
pool_fake_y = ImagePool(pool_size=50)

LAMBDA_CYCLE = 10.0
LAMBDA_IDENTITY = 5.0
loss_obj = tf.keras.losses.BinaryCrossentropy(from_logits=True)

def discriminator_loss(real, generated):
    real_loss = loss_obj(tf.fill(tf.shape(real), 0.9), real)
    generated_loss = loss_obj(tf.zeros_like(generated), generated)
    return (real_loss + generated_loss) * 0.5

def generator_loss(generated):
    return loss_obj(tf.ones_like(generated), generated)

def calc_cycle_loss(real_image, cycled_image):
    return LAMBDA_CYCLE * tf.reduce_mean(tf.abs(real_image - cycled_image))

def calc_identity_loss(real_image, same_image):
    return LAMBDA_IDENTITY * tf.reduce_mean(tf.abs(real_image - same_image))

INITIAL_LR = 2e-4
generator_g_optimizer = tf.keras.optimizers.Adam(INITIAL_LR, beta_1=0.5)
generator_f_optimizer = tf.keras.optimizers.Adam(INITIAL_LR, beta_1=0.5)
discriminator_x_optimizer = tf.keras.optimizers.Adam(INITIAL_LR, beta_1=0.5)
discriminator_y_optimizer = tf.keras.optimizers.Adam(INITIAL_LR, beta_1=0.5)

def update_learning_rate(epoch, total_epochs, decay_start_epoch=20):
    if epoch >= decay_start_epoch:
        factor = 1.0 - (epoch - decay_start_epoch) / float(max(total_epochs - decay_start_epoch, 1))
        new_lr = max(INITIAL_LR * factor, 1e-6)
        generator_g_optimizer.learning_rate.assign(new_lr)
        generator_f_optimizer.learning_rate.assign(new_lr)
        discriminator_x_optimizer.learning_rate.assign(new_lr)
        discriminator_y_optimizer.learning_rate.assign(new_lr)

def generate_and_display_images(model_g, model_f, test_input_day, test_input_night, epoch=None):
    fake_night = model_g(test_input_day, training=False)
    cycled_day = model_f(fake_night, training=False)

    fake_day = model_f(test_input_night, training=False)
    cycled_night = model_g(fake_day, training=False)

    plt.figure(figsize=(15, 6))

    imgs_1 = [test_input_day[0], fake_night[0], cycled_day[0]]
    titles_1 = ["Día Real (Input A)", "Noche Sintetizada (G(A))", "Día Reconstruido (F(G(A)))"]
    for i in range(3):
        plt.subplot(2, 3, i + 1)
        plt.title(titles_1[i], fontsize=10)
        plt.imshow(np.clip(imgs_1[i].numpy() * 0.5 + 0.5, 0.0, 1.0))
        plt.axis('off')

    imgs_2 = [test_input_night[0], fake_day[0], cycled_night[0]]
    titles_2 = ["Noche Real (Input B)", "Día Sintetizado (F(B))", "Noche Reconstruida (G(F(B)))"]
    for i in range(3):
        plt.subplot(2, 3, i + 4)
        plt.title(titles_2[i], fontsize=10)
        plt.imshow(np.clip(imgs_2[i].numpy() * 0.5 + 0.5, 0.0, 1.0))
        plt.axis('off')

    if epoch is not None:
        plt.suptitle(f"Resultados de Validación - Época {epoch}", fontsize=14)
    plt.tight_layout()
    plt.show()

## 4. Bucle de Entrenamiento Optimizado (45 Épocas), TensorBoard y Checkpoints

El entrenamiento está configurado para un máximo de **45 épocas** con decaimiento de *learning rate* desde la época 20. Para acelerar la ejecución en GPU T4 sin alterar ni un solo cálculo matemático del modelo (0% impacto en la calidad), se aplican **tres optimizaciones sin pérdida**:
1. **Separación en dos pasos compilados (`@tf.function`):** `train_generators_step` actualiza `G` y `F` y devuelve las imágenes generadas `fake_x` y `fake_y`; inmediatamente después se consulta el `ImagePool` y `train_discriminators_step` actualiza `Dx` y `Dy`. Esto es matemáticamente equivalente al paso unificado (los discriminadores aún conservan sus pesos previos cuando evalúan `G` y `F`), pero elimina el doble *forward pass* en modo *eager* que antes recalculaba los generadores solo para llenar el buffer.
2. **Pre-construcción del pipeline emparejado:** `tf.data.Dataset.zip((train_dia, train_noche)).prefetch(AUTOTUNE)` se instancia una sola vez fuera del ciclo de épocas, manteniendo el *shuffle* y el *random jittering* independientes en cada época.
3. **Menor sincronización GPU $\to$ CPU:** Las pérdidas se acumulan como tensores y se promedian una sola vez al cierre de cada época, actualizando la barra `tqdm` cada 20 pasos.

### Monitoreo en TensorBoard e Historial
En cada época se registran en `logs/cyclegan_<fecha>` las métricas escalares (`G_loss`, `D_loss`, `Cycle_loss`, `Identity_loss` y `Learning_Rate`) mediante `tf.summary`, además de imágenes de validación cada 5 épocas y checkpoints de respaldo.

In [ ]:
checkpoint_dir = "./checkpoints/cyclegan_day2night"
ckpt = tf.train.Checkpoint(
    generator_g=generator_g,
    generator_f=generator_f,
    discriminator_x=discriminator_x,
    discriminator_y=discriminator_y,
    generator_g_optimizer=generator_g_optimizer,
    generator_f_optimizer=generator_f_optimizer,
    discriminator_x_optimizer=discriminator_x_optimizer,
    discriminator_y_optimizer=discriminator_y_optimizer
)
ckpt_manager = tf.train.CheckpointManager(ckpt, checkpoint_dir, max_to_keep=3)

log_dir = os.path.join("logs", "cyclegan_" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S"))
tb_writer = tf.summary.create_file_writer(log_dir)

@tf.function
def train_generators_step(real_x, real_y):
    with tf.GradientTape(persistent=True) as tape:
        fake_y = generator_g(real_x, training=True)
        cycled_x = generator_f(fake_y, training=True)

        fake_x = generator_f(real_y, training=True)
        cycled_y = generator_g(fake_x, training=True)

        same_x = generator_f(real_x, training=True)
        same_y = generator_g(real_y, training=True)

        disc_fake_x = discriminator_x(fake_x, training=True)
        disc_fake_y = discriminator_y(fake_y, training=True)

        gen_g_loss = generator_loss(disc_fake_y)
        gen_f_loss = generator_loss(disc_fake_x)

        total_cycle_loss = calc_cycle_loss(real_x, cycled_x) + calc_cycle_loss(real_y, cycled_y)
        id_loss_g = calc_identity_loss(real_y, same_y)
        id_loss_f = calc_identity_loss(real_x, same_x)

        total_gen_g_loss = gen_g_loss + total_cycle_loss + id_loss_g
        total_gen_f_loss = gen_f_loss + total_cycle_loss + id_loss_f

    gen_g_grads = tape.gradient(total_gen_g_loss, generator_g.trainable_variables)
    gen_f_grads = tape.gradient(total_gen_f_loss, generator_f.trainable_variables)
    del tape

    generator_g_optimizer.apply_gradients(zip(gen_g_grads, generator_g.trainable_variables))
    generator_f_optimizer.apply_gradients(zip(gen_f_grads, generator_f.trainable_variables))

    return {
        "gen_loss": (total_gen_g_loss + total_gen_f_loss) * 0.5,
        "cycle_loss": total_cycle_loss,
        "identity_loss": (id_loss_g + id_loss_f) * 0.5,
        "fake_x": fake_x,
        "fake_y": fake_y
    }

@tf.function
def train_discriminators_step(real_x, real_y, fake_x_buffered, fake_y_buffered):
    with tf.GradientTape(persistent=True) as tape:
        disc_real_x = discriminator_x(real_x, training=True)
        disc_real_y = discriminator_y(real_y, training=True)
        disc_fake_buf_x = discriminator_x(fake_x_buffered, training=True)
        disc_fake_buf_y = discriminator_y(fake_y_buffered, training=True)

        disc_x_loss = discriminator_loss(disc_real_x, disc_fake_buf_x)
        disc_y_loss = discriminator_loss(disc_real_y, disc_fake_buf_y)

    disc_x_grads = tape.gradient(disc_x_loss, discriminator_x.trainable_variables)
    disc_y_grads = tape.gradient(disc_y_loss, discriminator_y.trainable_variables)
    del tape

    discriminator_x_optimizer.apply_gradients(zip(disc_x_grads, discriminator_x.trainable_variables))
    discriminator_y_optimizer.apply_gradients(zip(disc_y_grads, discriminator_y.trainable_variables))

    return (disc_x_loss + disc_y_loss) * 0.5

fixed_test_day = next(iter(test_dia))
fixed_test_night = next(iter(test_noche))

TOTAL_EPOCHS = 45
DECAY_START_EPOCH = 20
history = {"G_loss": [], "D_loss": [], "Cycle_loss": [], "Identity_loss": []}

train_ds = tf.data.Dataset.zip((train_dia, train_noche)).prefetch(tf.data.AUTOTUNE)
steps_total = min(n_train_a, n_train_b)

print(f"Iniciando entrenamiento por {TOTAL_EPOCHS} épocas con {steps_total} pasos por época...")
print(f"Logs de TensorBoard en: {log_dir}")

for epoch in range(1, TOTAL_EPOCHS + 1):
    start = time.time()
    update_learning_rate(epoch, TOTAL_EPOCHS, DECAY_START_EPOCH)
    current_lr = float(generator_g_optimizer.learning_rate.numpy())

    ep_g_losses = []
    ep_d_losses = []
    ep_c_losses = []
    ep_i_losses = []

    pbar = tqdm(train_ds, total=steps_total, desc=f"Época {epoch}/{TOTAL_EPOCHS}")

    for step, (real_x, real_y) in enumerate(pbar, start=1):
        gen_out = train_generators_step(real_x, real_y)

        fake_x_buffered = pool_fake_x.query(gen_out["fake_x"])
        fake_y_buffered = pool_fake_y.query(gen_out["fake_y"])

        d_loss = train_discriminators_step(real_x, real_y, fake_x_buffered, fake_y_buffered)

        ep_g_losses.append(gen_out["gen_loss"])
        ep_d_losses.append(d_loss)
        ep_c_losses.append(gen_out["cycle_loss"])
        ep_i_losses.append(gen_out["identity_loss"])

        if step % 20 == 0 or step == steps_total:
            pbar.set_postfix({
                "G": f"{float(gen_out['gen_loss']):.2f}",
                "D": f"{float(d_loss):.2f}",
                "Cycle": f"{float(gen_out['cycle_loss']):.2f}",
                "LR": f"{current_lr:.1e}"
            })

    mean_g = float(tf.reduce_mean(ep_g_losses))
    mean_d = float(tf.reduce_mean(ep_d_losses))
    mean_c = float(tf.reduce_mean(ep_c_losses))
    mean_i = float(tf.reduce_mean(ep_i_losses))

    history["G_loss"].append(mean_g)
    history["D_loss"].append(mean_d)
    history["Cycle_loss"].append(mean_c)
    history["Identity_loss"].append(mean_i)

    with tb_writer.as_default():
        tf.summary.scalar("Perdidas/G_loss", mean_g, step=epoch)
        tf.summary.scalar("Perdidas/D_loss", mean_d, step=epoch)
        tf.summary.scalar("Perdidas/Cycle_loss", mean_c, step=epoch)
        tf.summary.scalar("Perdidas/Identity_loss", mean_i, step=epoch)
        tf.summary.scalar("Hiperparametros/Learning_Rate", current_lr, step=epoch)

        if epoch % 5 == 0 or epoch == 1 or epoch == TOTAL_EPOCHS:
            tb_fake_night = tf.clip_by_value((generator_g(fixed_test_day, training=False) + 1.0) * 0.5, 0.0, 1.0)
            tb_fake_day = tf.clip_by_value((generator_f(fixed_test_night, training=False) + 1.0) * 0.5, 0.0, 1.0)
            tf.summary.image("Validacion/Dia_a_Noche", tb_fake_night, step=epoch)
            tf.summary.image("Validacion/Noche_a_Dia", tb_fake_day, step=epoch)

    tb_writer.flush()

    if epoch % 5 == 0 or epoch == 1 or epoch == TOTAL_EPOCHS:
        clear_output(wait=True)
        generate_and_display_images(generator_g, generator_f, fixed_test_day, fixed_test_night, epoch=epoch)
        ckpt_path = ckpt_manager.save()
        print(f"Checkpoint guardado en: {ckpt_path}")

    print(f"Época {epoch}/{TOTAL_EPOCHS} completada en {time.time() - start:.1f}s | G_loss: {mean_g:.3f} | D_loss: {mean_d:.3f} | Cycle_loss: {mean_c:.3f} | Identity_loss: {mean_i:.3f}")

print("Entrenamiento finalizado exitosamente.")

## 5. Medidores de TensorBoard y Curvas de Convergencia
Esta sección integra el visor interactivo de **TensorBoard** con los registros generados durante el entrenamiento (`SCALARS` para `G_loss`, `D_loss`, `Cycle_loss`, `Identity_loss` y `Learning_Rate`, además de `IMAGES` para la evolución visual) y las gráficas estáticas de diagnóstico:
* **Convergencia saludable:** `Cycle_loss` e `Identity_loss` deben descender progresivamente (preservación geométrica y de color).
* **Equilibrio adversarial:** `D_loss` debe mantenerse estable entre $0.4$ y $0.7$ gracias al *Dropout* y al *Label Smoothing* (si $D_{loss} \rightarrow 0$, el discriminador habría ganado y sobreajustado).

In [ ]:
%load_ext tensorboard
%tensorboard --logdir logs

In [ ]:
epochs_range = range(1, len(history["G_loss"]) + 1)

plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
plt.plot(epochs_range, history["G_loss"], label='Pérdida Generadores (G+F)/2', color='crimson')
plt.plot(epochs_range, history["D_loss"], label='Pérdida Discriminadores (Dx+Dy)/2', color='royalblue')
plt.title('Dinámica Adversarial Generadores vs Discriminadores')
plt.xlabel('Época')
plt.ylabel('Pérdida')
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(epochs_range, history["Cycle_loss"], label='Pérdida de Consistencia Cíclica', color='darkorange')
plt.plot(epochs_range, history["Identity_loss"], label='Pérdida de Identidad (G+F)/2', color='seagreen')
plt.title('Fidelidad Estructural y Cromática')
plt.xlabel('Época')
plt.ylabel('Pérdida L1')
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend()

plt.tight_layout()
plt.show()

## 6. Evaluación Cuantitativa y Análisis Estadístico de Convergencia

Para complementar la inspección visual sobre el conjunto de prueba inédito (`testA` y `testB`), se realiza una evaluación objetiva en dos niveles:

1. **Análisis por Fases de Entrenamiento (45 Épocas):** Se segmenta el historial de pérdidas (`history`) en tres etapas de 15 épocas cada una para cuantificar la estabilidad del equilibrio adversarial entre generadores y discriminadores, así como la reducción progresiva de la pérdida de consistencia cíclica ($\mathcal{L}_{cycle}$) tras activar el decaimiento de la tasa de aprendizaje (*LR Decay* desde la época 20).
2. **Métricas de Fidelidad Estructural y Reconstrucción Cíclica:** Utilizando ambos generadores entrenados ($G: \text{Día} \to \text{Noche}$ y $F: \text{Noche} \to \text{Día}$), se evalúan los dos ciclos completos ($F(G(x)) \approx x$ sobre `testA` y $G(F(y)) \approx y$ sobre `testB`) mediante:
   * **SSIM (*Structural Similarity Index Measure*):** Mide la preservación de estructuras espaciales, bordes y geometría urbana (rango $[0, 1]$, ideal cercano a $1.0$).
   * **PSNR (*Peak Signal-to-Noise Ratio*):** Cuantifica la fidelidad de reconstrucción a nivel de píxel en decibelios ($\text{dB}$).
   * **Error $L_1$ de Ciclo en Prueba:** Verifica que la pérdida de reconstrucción en datos nunca vistos sea consistente con la observada al final del entrenamiento.

In [ ]:
keys_disponibles = list(history.keys())
df_history = pd.DataFrame(history)
df_history['Epoca'] = np.arange(1, len(df_history) + 1)

total_epocas = len(df_history)
bins = [0, total_epocas // 3, (2 * total_epocas) // 3, total_epocas]
labels = [
    f"Fase 1 (Épocas 1-{bins[1]}): Exploración",
    f"Fase 2 (Épocas {bins[1]+1}-{bins[2]}): Estabilización",
    f"Fase 3 (Épocas {bins[2]+1}-{bins[3]}): Refinamiento (LR Decay)"
]
df_history['Fase'] = pd.cut(df_history['Epoca'], bins=bins, labels=labels)

resumen_fases = df_history.groupby('Fase', observed=False)[keys_disponibles].agg(['mean', 'std'])
print("=== Resumen Estadístico por Fases de Entrenamiento ===")
display(resumen_fases.round(4))

if 'Cycle_loss' in df_history.columns and len(df_history) > 0:
    c_ini = df_history['Cycle_loss'].iloc[0]
    c_fin = df_history['Cycle_loss'].iloc[-1]
    mejora_pct = ((c_ini - c_fin) / c_ini) * 100
    print(f"\nReducción total de L1 Cycle Loss: {c_ini:.4f} -> {c_fin:.4f} (-{mejora_pct:.2f}%)")

def evaluar_ciclo_dataset(dataset, gen_forward, gen_backward):
    ssim_list, psnr_list, l1_list = [], [], []
    for batch in dataset:
        fake = gen_forward(batch, training=False)
        rec = gen_backward(fake, training=False)

        img_real_01 = tf.clip_by_value((batch + 1.0) / 2.0, 0.0, 1.0)
        img_rec_01 = tf.clip_by_value((rec + 1.0) / 2.0, 0.0, 1.0)

        ssim_val = tf.image.ssim(img_real_01, img_rec_01, max_val=1.0).numpy()
        psnr_val = tf.image.psnr(img_real_01, img_rec_01, max_val=1.0).numpy()
        l1_val = tf.reduce_mean(tf.abs(batch - rec)).numpy()

        ssim_list.extend(np.atleast_1d(ssim_val))
        psnr_list.extend(np.atleast_1d(psnr_val))
        l1_list.append(l1_val)

    return pd.DataFrame({
        'Métrica': ['SSIM (Similitud Estructural)', 'PSNR (Relación Señal/Ruido dB)', 'L1 Cycle Error (Test Inédito)'],
        'Promedio': [np.mean(ssim_list), np.mean(psnr_list), np.mean(l1_list)],
        'Desv. Estándar': [np.std(ssim_list), np.std(psnr_list), np.std(l1_list)],
        'Mínimo': [np.min(ssim_list), np.min(psnr_list), np.min(l1_list)],
        'Máximo': [np.max(ssim_list), np.max(psnr_list), np.max(l1_list)]
    })

print("\n=== Métricas Cuantitativas sobre testA (Ciclo Día -> Noche -> Día) ===")
display(evaluar_ciclo_dataset(test_dia, generator_g, generator_f).round(4))

print("\n=== Métricas Cuantitativas sobre testB (Ciclo Noche -> Día -> Noche) ===")
display(evaluar_ciclo_dataset(test_noche, generator_f, generator_g).round(4))

## 7. Evaluación Cualitativa sobre el Conjunto de Prueba Inédito (`testA`)
Evaluamos el modelo final sobre 8 imágenes de día inéditas de `testA` que la red **nunca vio durante el entrenamiento**, analizando:
1. Oscurecimiento coherente del cielo.
2. Iluminación nocturna y contraste en ventanas y farolas de edificios.
3. Preservación nítida de líneas y bordes urbanos.

In [ ]:
print("Evaluando generador en 8 imágenes inéditas de prueba (Día -> Noche)...")

test_samples = list(test_dia.take(8))
n_samples = len(test_samples)
plt.figure(figsize=(12, 3 * n_samples))

for idx, day_img in enumerate(test_samples):
    fake_night_img = generator_g(day_img, training=False)

    plt.subplot(n_samples, 2, idx * 2 + 1)
    plt.imshow(np.clip(day_img[0].numpy() * 0.5 + 0.5, 0.0, 1.0))
    plt.title(f"Test #{idx + 1} - Entrada Real (Día)")
    plt.axis('off')

    plt.subplot(n_samples, 2, idx * 2 + 2)
    plt.imshow(np.clip(fake_night_img[0].numpy() * 0.5 + 0.5, 0.0, 1.0))
    plt.title(f"Test #{idx + 1} - Generada por CycleGAN (Noche)")
    plt.axis('off')

plt.tight_layout()
plt.show()

## 8. Exportación y Guardado de Ambos Modelos Entrenados
Guardamos los dos generadores (`generator_g` para **Día $\rightarrow$ Noche** y `generator_f` para **Noche $\rightarrow$ Día**) en formato nativo Keras (`.keras`) y en formato tradicional HDF5 (`.h5`) dentro de `modelos_exportados/` para su descarga o despliegue posterior.

In [ ]:
os.makedirs("modelos_exportados", exist_ok=True)

generator_g.save("modelos_exportados/cyclegan_dia_a_noche.keras")
generator_g.save("modelos_exportados/cyclegan_dia_a_noche.h5")

generator_f.save("modelos_exportados/cyclegan_noche_a_dia.keras")
generator_f.save("modelos_exportados/cyclegan_noche_a_dia.h5")

print("Generadores exportados exitosamente en modelos_exportados/:")
print(" - cyclegan_dia_a_noche.keras / .h5")
print(" - cyclegan_noche_a_dia.keras / .h5")

## 9. Presentación Final: Pruebas con Fotografías de Autoría Propia
Esta sección corresponde a la **presentación final del proyecto**, aplicando ambos generadores entrenados ($G: \text{Día} \to \text{Noche}$ y $F: \text{Noche} \to \text{Día}$) sobre fotografías reales de autoría propia ubicadas en `prueba final/dia` y `prueba final/noche`.

* **Corrección de orientación EXIF (`ImageOps.exif_transpose`):** Las fotos tomadas con celular o compartidas por WhatsApp pueden almacenar la rotación en metadatos EXIF en lugar de los píxeles. Al decodificarlas con Pillow y aplicar `exif_transpose`, se garantiza que todas las fotografías aparezcan correctamente orientadas antes de redimensionarlas a `256x256` y normalizarlas en $[-1, 1]$.
* **Respaldo de carga:** Si los generadores no estuvieran en memoria, se cargan automáticamente desde `modelos_exportados/`.
* **Salida:** Se presenta la comparación directa **Original vs. Generada** y se exportan las imágenes resultantes en `prueba final/resultados/`.

In [ ]:
base_prueba = os.path.join(REPO_DIR, "prueba final")
ruta_dia = os.path.join(base_prueba, "dia")
ruta_noche = os.path.join(base_prueba, "noche")
ruta_resultados = os.path.join(base_prueba, "resultados")
os.makedirs(ruta_resultados, exist_ok=True)

gen_dia_a_noche = globals().get('generator_g') or globals().get('G') or globals().get('gen_A2B')
gen_noche_a_dia = globals().get('generator_f') or globals().get('F') or globals().get('gen_B2A')

if gen_dia_a_noche is None and os.path.exists("modelos_exportados/cyclegan_dia_a_noche.keras"):
    gen_dia_a_noche = tf.keras.models.load_model("modelos_exportados/cyclegan_dia_a_noche.keras")
if gen_noche_a_dia is None and os.path.exists("modelos_exportados/cyclegan_noche_a_dia.keras"):
    gen_noche_a_dia = tf.keras.models.load_model("modelos_exportados/cyclegan_noche_a_dia.keras")

if gen_dia_a_noche is None or gen_noche_a_dia is None:
    raise NameError("No se encontraron los generadores en memoria ni en modelos_exportados/. Ejecuta el entrenamiento o exporta los modelos primero.")

def cargar_y_preprocesar(ruta_imagen):
    with Image.open(ruta_imagen) as img:
        img = ImageOps.exif_transpose(img).convert("RGB")
        img_resized_pil = img.resize((256, 256), Image.Resampling.BICUBIC)
    img_arr = np.asarray(img_resized_pil, dtype=np.float32)
    img_norm = (img_arr / 127.5) - 1.0
    return tf.expand_dims(img_norm, 0), img_arr.astype(np.uint8)

def guardar_imagen_generada(tensor_salida, ruta_destino):
    img_array = (tensor_salida[0].numpy() + 1.0) * 127.5
    img_array = np.clip(img_array, 0, 255).astype(np.uint8)
    Image.fromarray(img_array).save(ruta_destino)
    return img_array

extensiones = ('*.jpg', '*.jpeg', '*.png', '*.webp', '*.JPG', '*.JPEG', '*.PNG', '*.WEBP')
imagenes_dia = sorted({f for ext in extensiones for f in glob.glob(os.path.join(ruta_dia, ext))})
imagenes_noche = sorted({f for ext in extensiones for f in glob.glob(os.path.join(ruta_noche, ext))})

print(f"Carpeta de pruebas finales: {base_prueba}")
print(f"Imágenes encontradas -> Día: {len(imagenes_dia)} | Noche: {len(imagenes_noche)}")

for ruta_img in imagenes_dia:
    nombre_base = os.path.splitext(os.path.basename(ruta_img))[0]
    tensor_in, img_orig_256 = cargar_y_preprocesar(ruta_img)
    prediccion = gen_dia_a_noche(tensor_in, training=False)

    ruta_guardado = os.path.join(ruta_resultados, f"resultado_dia_a_noche_{nombre_base}.png")
    img_generada = guardar_imagen_generada(prediccion, ruta_guardado)

    fig, axes = plt.subplots(1, 2, figsize=(12, 6), dpi=150)
    axes[0].imshow(img_orig_256)
    axes[0].set_title(f"Original (Día): {nombre_base}", fontsize=12)
    axes[0].axis("off")

    axes[1].imshow(img_generada)
    axes[1].set_title("Transformación CycleGAN (Día → Noche)", fontsize=12)
    axes[1].axis("off")
    plt.tight_layout()
    plt.show()
    print(f"Guardada en: {ruta_guardado}")

for ruta_img in imagenes_noche:
    nombre_base = os.path.splitext(os.path.basename(ruta_img))[0]
    tensor_in, img_orig_256 = cargar_y_preprocesar(ruta_img)
    prediccion = gen_noche_a_dia(tensor_in, training=False)

    ruta_guardado = os.path.join(ruta_resultados, f"resultado_noche_a_dia_{nombre_base}.png")
    img_generada = guardar_imagen_generada(prediccion, ruta_guardado)

    fig, axes = plt.subplots(1, 2, figsize=(12, 6), dpi=150)
    axes[0].imshow(img_orig_256)
    axes[0].set_title(f"Original (Noche): {nombre_base}", fontsize=12)
    axes[0].axis("off")

    axes[1].imshow(img_generada)
    axes[1].set_title("Transformación CycleGAN (Noche → Día)", fontsize=12)
    axes[1].axis("off")
    plt.tight_layout()
    plt.show()
    print(f"Guardada en: {ruta_guardado}")